# CertiScan: train the layout detector and evaluate the full pipeline (Colab, T4 GPU)

**Runtime → Change runtime type → T4 GPU**, then run all cells in order (about 45–60 min).

1. Generate 600 synthetic Material Test Reports with exact labels
2. Fine-tune **YOLOv8n** to detect 7 layout regions (optional **RT-DETR** comparison)
3. Run an evaluation matrix that separates OCR, layout and parser errors
4. Save the weights and results to Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/Harshchoubey55/certiscan.git"
QUICK = False   # True: smaller run (300 pages, 15 epochs, 30-page eval) for a fast check

import os
os.chdir("/content")
if not os.path.exists("certiscan"):
    !git clone -q {REPO_URL} certiscan
os.chdir("/content/certiscan")
!git pull -q
!ls

In [ ]:
!pip install -q -r requirements.txt -r requirements-ml.txt
!apt-get -qq install -y tesseract-ocr > /dev/null
!python -m pytest -q

## 2. Synthetic dataset
70/15/15 train/val/test split. Every page includes YOLO labels, word boxes and the ground-truth record.

In [ ]:
!python -m certiscan.synth.generate --out data/synth --n {300 if QUICK else 600} --seed 42 --workers 2

In [ ]:
import json, cv2, matplotlib.pyplot as plt
from certiscan.types import REGION_CLASSES

def show(i=0, split="train"):
    gt = json.load(open(f"data/synth/gt/{split}/mtr_{i:05d}.json"))
    img = cv2.cvtColor(cv2.imread(f"data/synth/images/{split}/{gt['id']}.jpg"), cv2.COLOR_BGR2RGB)
    for r in gt["regions"]:
        x0, y0, x1, y1 = map(int, r["box"])
        cv2.rectangle(img, (x0, y0), (x1, y1), (255, 0, 0), 4)
        cv2.putText(img, r["cls"], (x0, y0 - 8), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    plt.figure(figsize=(8, 10)); plt.imshow(img); plt.axis("off")
    print(json.dumps(gt["fields"], indent=1)[:1500])

show(3)

## 3. Experiment tracking (optional)
Logs to W&B if `WANDB_API_KEY` is set in Colab Secrets; otherwise training runs without it.

In [ ]:
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    !yolo settings wandb=True
except Exception:
    !yolo settings wandb=False

## 4. Fine-tune YOLOv8n for layout detection
`fliplr=0`: a mirrored document is not a valid document. Mosaic is reduced because the regions are page-sized.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(data="data/synth/data.yaml", epochs=15 if QUICK else 40, imgsz=640, batch=16, fliplr=0.0, mosaic=0.3,
            degrees=0.0, hsv_h=0.0, project="runs", name="yolov8n_layout", exist_ok=True, seed=0, plots=True)

In [ ]:
import glob
# newer Ultralytics releases nest `project` under runs/detect/, so locate the run instead of hard-coding it
RUN = sorted(glob.glob("runs/**/yolov8n_layout", recursive=True))[-1]
best = YOLO(f"{RUN}/weights/best.pt")
m = best.val(data="data/synth/data.yaml", split="test", imgsz=640, plots=True)
print(f"test mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}")
for i, c in enumerate(REGION_CLASSES):
    print(f"  {c:12s} AP50-95 = {m.box.maps[i]:.3f}")

### (Optional, ~25 min) RT-DETR comparison
A transformer detector on the same data. Skip this if you're short on time.

In [ ]:
RUN_RTDETR = False
if RUN_RTDETR:
    from ultralytics import RTDETR
    rt = RTDETR("rtdetr-l.pt")
    rt.train(data="data/synth/data.yaml", epochs=25, imgsz=640, batch=8, fliplr=0.0, mosaic=0.3,
             project="runs", name="rtdetr_layout", exist_ok=True, seed=0)
    mr = RTDETR(rt.trainer.best).val(data="data/synth/data.yaml", split="test", imgsz=640)
    print(f"RT-DETR test mAP50 = {mr.box.map50:.3f}   mAP50-95 = {mr.box.map:.3f}")

## 5. Evaluation matrix
Swapping one component at a time shows where the remaining errors come from:
parser (oracle text + true layout), layout (keyword rules vs YOLO), and OCR engine (EasyOCR vs Tesseract).

In [ ]:
W = f"{RUN}/weights/best.pt"
LIM = "--limit 30" if QUICK else ""
!python scripts/evaluate.py --data data/synth --ocr oracle    --layout gt {LIM}
!python scripts/evaluate.py --data data/synth --ocr oracle    --layout rules {LIM}
!python scripts/evaluate.py --data data/synth --ocr easyocr   --layout gt {LIM}
if not QUICK:
    !python scripts/evaluate.py --data data/synth --ocr tesseract --layout gt
!python scripts/evaluate.py --data data/synth --ocr easyocr   --layout rules {LIM}
!python scripts/evaluate.py --data data/synth --ocr easyocr   --layout yolo --weights {W} {LIM}
if not QUICK:
    !python scripts/evaluate.py --data data/synth --ocr tesseract --layout yolo --weights {W}

In [ ]:
!python scripts/summarize.py results
from IPython.display import Markdown, display
display(Markdown(open("results/RESULTS.md").read()))

## 6. Error analysis: what does the full system get wrong?

In [ ]:
from collections import Counter
d = json.load(open("results/eval_easyocr_yolo_test.json"))
miss = Counter(k.split("@")[0] for doc in d["documents"] for k, _ in doc["missed"])
print("Most-missed fields:", miss.most_common(10))
for doc in [x for x in d["documents"] if not x["exact"]][:5]:
    print(doc["id"], "| missed:", doc["missed"][:4], "| got:", doc["spurious"][:4])

## 7. Save artifacts to Google Drive
Download `layout_best.pt` into `models/` in your local copy to run the web app.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/certiscan
!cp {RUN}/weights/best.pt /content/drive/MyDrive/certiscan/layout_best.pt
!cp -r results /content/drive/MyDrive/certiscan/
!cp {RUN}/results.png {RUN}/confusion_matrix.png /content/drive/MyDrive/certiscan/ 2>/dev/null
!ls -la /content/drive/MyDrive/certiscan